<a href="https://colab.research.google.com/github/luonghoangvietnb-star/Hadoop-Ecommerce-Lab/blob/main/Hadoop_Ecommerce_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Kiểm tra Java
!java -version

# Cài đặt Java nếu chưa có
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless

# Tải Apache Hadoop 3.5.0
!wget -q --show-progress https://archive.apache.org/dist/hadoop/common/hadoop-3.5.0/hadoop-3.5.0.tar.gz

# Giải nén Hadoop
!tar -xzf hadoop-3.5.0.tar.gz

[0.025s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.025s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  openjdk-17-jre-headless
Suggested packages:
  openjdk-17-demo openjdk-17-source libnss-mdns fonts-dejavu-extra
  fonts-ipafont-gothic fonts-ipa

In [2]:
import os
import subprocess

os.environ["HADOOP_HOME"] = "/content/hadoop-3.5.0"
os.environ["JAVA_HOME"] = os.path.realpath(
    subprocess.check_output(
        ["bash", "-lc", "dirname $(dirname $(readlink -f $(which java)))"],
        text=True
    ).strip()
)
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "/bin:" + os.environ["PATH"]

with open(os.environ["HADOOP_HOME"] + "/etc/hadoop/hadoop-env.sh", "a") as f:
    f.write("\nexport JAVA_HOME=" + os.environ["JAVA_HOME"] + "\n")

!hadoop version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Hadoop 3.5.0
Source code repository https://github.com/apache/hadoop.git -r dbcc7cd797100e6b32cd84f85b53a5193a5f9af0
Compiled by cnauroth on 2026-03-24T16:56Z
Compiled on platform linux-x86_64
Compiled with protoc 3.25.5
From source with checksum 64582ed0a62a169c39a19c734a8f33b5
This command was run using /content/hadoop-3.5.0/share/hadoop/common/hadoop-common-3.5.0.jar


In [3]:
%%writefile orders_2026_09.csv
transaction_id,order_date,order_time,province,category,channel,quantity,unit_price,amount,status,shipping_days
TX001,2026-09-01,08:30,Ha Noi,Electronics,Web,1,3400000,3400000,SUCCESS,2
TX002,2026-09-01,09:00,Da Nang,Fashion,App,2,260000,520000,SUCCESS,3
TX003,2026-09-02,10:30,Can Tho,Grocery,Web,1,180000,180000,FAILED,4
TX004,2026-09-02,11:00,Lam Dong,Book,App,1,250000,250000,SUCCESS,2
TX005,2026-09-03,13:30,Ha Noi,Electronics,Web,1,2500000,2500000,SUCCESS,1

Writing orders_2026_09.csv


In [4]:
!head orders_2026_09.csv

transaction_id,order_date,order_time,province,category,channel,quantity,unit_price,amount,status,shipping_days
TX001,2026-09-01,08:30,Ha Noi,Electronics,Web,1,3400000,3400000,SUCCESS,2
TX002,2026-09-01,09:00,Da Nang,Fashion,App,2,260000,520000,SUCCESS,3
TX003,2026-09-02,10:30,Can Tho,Grocery,Web,1,180000,180000,FAILED,4
TX004,2026-09-02,11:00,Lam Dong,Book,App,1,250000,250000,SUCCESS,2
TX005,2026-09-03,13:30,Ha Noi,Electronics,Web,1,2500000,2500000,SUCCESS,1


In [5]:
%%writefile mapper_revenue.py
import sys
import csv

reader = csv.DictReader(sys.stdin)
for row in reader:
    if row["status"] == "SUCCESS":
        category = row["category"]
        amount = int(row["amount"])
        print(f"{category}\t{amount}")

Writing mapper_revenue.py


In [6]:
%%writefile reducer_revenue.py
import sys

current = None
total = 0

for line in sys.stdin:
    category, amount = line.strip().split("\t")
    amount = int(amount)

    if current == category:
        total += amount
    else:
        if current is not None:
            print(f"{current}\t{total}")
        current = category
        total = amount

if current is not None:
    print(f"{current}\t{total}")

Writing reducer_revenue.py


In [7]:
!cat orders_2026_09.csv | python3 mapper_revenue.py | sort | python3 reducer_revenue.py

Book	250000
Electronics	5900000
Fashion	520000


In [8]:
import glob
import shutil
import subprocess

# Xóa thư mục output cũ nếu có
shutil.rmtree("/content/output_revenue", ignore_errors=True)

# Lấy file jar streaming
jar = glob.glob(
    "/content/hadoop-3.5.0/share/hadoop/tools/lib/hadoop-streaming*.jar"
)[0]

cmd = [
    "hadoop", "jar", jar,
    "-D", "mapreduce.framework.name=local",
    "-D", "fs.defaultFS=file:///",
    "-files", "/content/mapper_revenue.py,/content/reducer_revenue.py",
    "-mapper", "python3 mapper_revenue.py",
    "-reducer", "python3 reducer_revenue.py",
    "-input", "file:///content/orders_2026_09.csv",
    "-output", "file:///content/output_revenue"
]

subprocess.run(cmd, check=True)

CompletedProcess(args=['hadoop', 'jar', '/content/hadoop-3.5.0/share/hadoop/tools/lib/hadoop-streaming-3.5.0.jar', '-D', 'mapreduce.framework.name=local', '-D', 'fs.defaultFS=file:///', '-files', '/content/mapper_revenue.py,/content/reducer_revenue.py', '-mapper', 'python3 mapper_revenue.py', '-reducer', 'python3 reducer_revenue.py', '-input', 'file:///content/orders_2026_09.csv', '-output', 'file:///content/output_revenue'], returncode=0)

In [9]:
!ls -l /content/output_revenue
!cat /content/output_revenue/part-*

total 4
-rw-r--r-- 1 root root 47 Oct 14 03:29 part-00000
-rw-r--r-- 1 root root  0 Oct 14 03:29 _SUCCESS
Book	250000
Electronics	5900000
Fashion	520000


In [10]:
!sort -k2,2nr /content/output_revenue/part-* | head -3

Electronics	5900000
Fashion	520000
Book	250000
